# 02 · The N-Hop Join Problem

The first showpiece. "Who is connected to whom" is easy at one hop (a single join) and
turns ugly fast:

- **1 hop** — one join.
- **4 hops** — four stacked self-joins.
- **variable length (1..6)** — a *recursive CTE with a visited-set guard* so it can't
  loop forever on a cycle. Twenty-plus lines that are hard to get right.

In Cypher through AGE it stays one line: `MATCH (a)-[:KNOWS*1..6]->(b)`. We measure lines
of code, and verify AGE, SQL, and NetworkX agree on exactly which accounts are reachable.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt

from src.age.connect import age_available, get_connection
from src.baselines.sql_recursive import _REACHABLE
from src.config import load_settings
from src.datasets.graphs import KNOWS, all_graphs
from src.evaluation.correctness import sets_agree
from src.evaluation.readability import count_loc
from src.outputs import figure_path, save_json
from src.queries import traversal
from src.queries.traversal import variable_length_cypher
from src.visualisation import draw_graph

settings = load_settings()
AGE = age_available(settings)
social = all_graphs(settings)["social_txn"]
conn = get_connection(settings) if AGE else None
START = 0
print("AGE:", AGE, "| start account:", START)

AGE: True | start account: 0


## Reachability at growing depth — and do the three approaches agree?

For each depth we compute the set of reachable accounts three ways and assert they are
identical (`AGE == SQL == NetworkX`), which is the correctness bar for this chapter.

One honest subtlety: openCypher's `*1..d` forbids reusing a *relationship* but allows
revisiting a *node*. Because `KNOWS` is reciprocal, AGE therefore also returns the start
account via a there-and-back walk, while our baselines enumerate simple paths. The only
difference is the start itself, so we compare the sets with the start removed.

In [2]:
depths = [1, 2, 3, 4, 5, 6]
reach = {}
for d in depths:
    nx_set = traversal.reachable_networkx(social, START, 1, d, KNOWS)
    sql_set = traversal.reachable_sql(social, START, 1, d, KNOWS)
    age_set = (
        traversal.reachable_age(conn, settings.graphs.social, START, 1, d, KNOWS)
        if AGE else set(nx_set)
    )
    age_cmp = age_set - {START}  # drop the start (AGE's there-and-back walk; see note above)
    agree = sets_agree(nx_set, sql_set, age_cmp)
    assert agree, f"approaches disagree at depth {d}"
    reach[d] = {"reachable": len(nx_set), "agree": agree}
    print(f"depth {d}: {len(nx_set):>3} accounts reachable | AGE==SQL==NetworkX: {agree}")

reachability = {
    "graph": "social_txn",
    "start": START,
    "one_hop": reach[1]["reachable"],
    "four_hop": reach[4]["reachable"],
    "var_1_to_6": reach[6]["reachable"],
    "by_depth": reach,
    "all_approaches_agree": all(v["agree"] for v in reach.values()),
}
print("saved ->", save_json("02_reachability.json", reachability))

depth 1:   8 accounts reachable | AGE==SQL==NetworkX: True
depth 2:  54 accounts reachable | AGE==SQL==NetworkX: True
depth 3: 183 accounts reachable | AGE==SQL==NetworkX: True


depth 4: 362 accounts reachable | AGE==SQL==NetworkX: True


depth 5: 399 accounts reachable | AGE==SQL==NetworkX: True


depth 6: 399 accounts reachable | AGE==SQL==NetworkX: True
saved -> /workspaces/apache_age_101/outputs/data/02_reachability.json


## Read the three implementations — and count the lines

Same answer, wildly different effort. The SQL recursive CTE (with its visited-set guard)
is the real query from `src/baselines/sql_recursive.py`; the Cypher is one line.

In [3]:
cypher_q = variable_length_cypher(START, 1, 6, KNOWS)
sql_q = _REACHABLE.strip()
nx_lines = [
    "result, stack = set(), [(start, 0, frozenset({start}))]",
    "while stack:",
    "    node, depth, seen = stack.pop()",
    "    if min_hops <= depth <= max_hops: result.add(node)",
    "    if depth >= max_hops: continue",
    "    for nbr in G.successors(node):",
    "        if nbr not in seen: stack.append((nbr, depth + 1, seen | {nbr}))",
]
nx_q = "\n".join(nx_lines)

loc = {
    "AGE (Cypher)": count_loc(cypher_q),
    "SQL (recursive CTE)": count_loc(sql_q),
    "NetworkX": count_loc(nx_q),
}
print("Lines of code per approach:", json.dumps(loc, indent=2))
print("\nThe one-line Cypher:\n ", cypher_q)
save_json("02_loc.json", loc)
save_json("02_queries.json", {"cypher": cypher_q, "sql": sql_q, "networkx": nx_q})

Lines of code per approach: {
  "AGE (Cypher)": 1,
  "SQL (recursive CTE)": 11,
  "NetworkX": 7
}

The one-line Cypher:
  MATCH (a:Account {node_id: 0})-[:KNOWS*1..6]->(b) RETURN DISTINCT b.node_id


PosixPath('/workspaces/apache_age_101/outputs/data/02_queries.json')

## Figure: reachability grows with depth

In [4]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, [reach[d]["reachable"] for d in depths], marker="o", color="#4c6ef5")
ax.set_xlabel("hop depth (variable-length *1..d)")
ax.set_ylabel("accounts reachable from start")
ax.set_title(f"Reachability vs depth (start = account {START})")
ax.grid(True, alpha=0.3)
fig.tight_layout()
p = figure_path("02_reachability_vs_depth.png")
fig.savefig(p, dpi=120)
plt.close(fig)
print("saved ->", p)
save_json("02_reachability_by_depth.json", {str(d): reach[d]["reachable"] for d in depths})

saved -> /workspaces/apache_age_101/outputs/figures/02_reachability_vs_depth.png


PosixPath('/workspaces/apache_age_101/outputs/data/02_reachability_by_depth.json')

## Figure: the reachable neighbourhood (what one Cypher line returns)

In [5]:
sub_ids = traversal.reachable_networkx(social, START, 1, 3, KNOWS) | {START}
sub = social.to_networkx(KNOWS).subgraph(sub_ids)
p = draw_graph(
    sub,
    figure_path("02_nhop_subgraph.png"),
    title=f"Within 3 hops of account {START} — one MATCH pattern",
    highlight={START},
)
print(f"{sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges -> {p}")

184 nodes, 1002 edges -> /workspaces/apache_age_101/outputs/figures/02_nhop_subgraph.png


In [6]:
if conn is not None:
    conn.close()
print("Done. Next: 03 — fraud rings and pathfinding.")

Done. Next: 03 — fraud rings and pathfinding.


**Takeaway.** The same traversal is one line of Cypher, a dozen-plus lines of
recursive SQL, and a short Python loop — and all three return the same accounts. Depth is
free in Cypher and expensive in SQL. Next: the question SQL effectively *cannot* answer.